In [ ]:

!pip install ultralytics opencv-python-headless pandas matplotlib seaborn


In [ ]:

import torch
import cv2
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("GPU Available:", torch.cuda.is_available())
print("YOLO imported successfully!")


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

import cv2
import os

video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)

# Extract video information
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = total_frames / fps if fps else 0

print("Video Name:", video_path)
print("FPS:", fps)
print("Total Frames:", total_frames)
print("Resolution:", width, "x", height)
print("Duration:", round(duration, 2), "seconds")

cap.release()


In [ ]:

import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)

# Read one frame
success, frame = cap.read()

if success:
    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 6))
    plt.imshow(frame_rgb)
    plt.title("Original Traffic Video Frame")
    plt.axis("off")
    plt.show()
else:
    print("Could not read video")

cap.release()


In [ ]:

from ultralytics import YOLO

# Load pretrained YOLO model
model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")


In [ ]:

import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)

success, frame = cap.read()

cap.release()

# Run YOLO detection
results = model(frame, conf=0.35)

# Draw bounding boxes
annotated_frame = results[0].plot()

# Display output
annotated_frame = cv2.cvtColor(
    annotated_frame, cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(14, 7))
plt.imshow(annotated_frame)
plt.title("YOLO Vehicle Detection")
plt.axis("off")
plt.show()


In [ ]:

# Vehicle class IDs
vehicle_classes = [2, 3, 5, 7]

results = model(
    frame,
    classes=vehicle_classes,
    conf=0.35
)

# Display vehicle-only detections
vehicle_frame = results[0].plot()

vehicle_frame = cv2.cvtColor(
    vehicle_frame, cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(14, 7))
plt.imshow(vehicle_frame)
plt.title("Vehicle-Only Detection")
plt.axis("off")
plt.show()


In [ ]:

result = results[0]

vehicle_counts = {
    "Car": 0,
    "Motorcycle": 0,
    "Bus": 0,
    "Truck": 0
}

class_names = model.names

for box in result.boxes:
    class_id = int(box.cls[0])
    class_name = class_names[class_id]

    if class_name == "car":
        vehicle_counts["Car"] += 1

    elif class_name == "motorcycle":
        vehicle_counts["Motorcycle"] += 1

    elif class_name == "bus":
        vehicle_counts["Bus"] += 1

    elif class_name == "truck":
        vehicle_counts["Truck"] += 1

print("Vehicle Count in One Frame:")
print(vehicle_counts)

print("Total Vehicles:",
      sum(vehicle_counts.values()))


In [ ]:

from ultralytics import YOLO
import cv2

# Load pretrained model
model = YOLO("yolo11n.pt")

# Vehicle class IDs
vehicle_classes = [2, 3, 5, 7]

# Open video
cap = cv2.VideoCapture(video_path)

# Tracking configuration
while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=vehicle_classes,
        conf=0.35,
        verbose=False
    )

    # Draw bounding boxes and tracking IDs
    annotated_frame = results[0].plot()

    # Display selected frame
    cv2_imshow(annotated_frame)

    break

cap.release()


In [ ]:

from google.colab.patches import cv2_imshow

print("cv2_imshow imported successfully!")


In [ ]:

results = model.track(
    frame,
    persist=True,
    tracker="bytetrack.yaml",
    classes=[2, 3, 5, 7],
    conf=0.35,
    verbose=False
)

annotated_frame = results[0].plot()

cv2_imshow(annotated_frame)


In [ ]:

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

print("Width:", width)
print("Height:", height)

# Horizontal counting line at 60% of video height
line_y = int(height * 0.60)

print("Counting line position:", line_y)


In [ ]:

from collections import defaultdict
import cv2

# Open video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output video
output_path = "vehicle_counting.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Counting line
line_y = int(height * 0.60)

# Vehicle classes
vehicle_classes = [2, 3, 5, 7]

class_names = model.names

# Store previous vehicle positions
previous_centers = {}

# Store IDs already counted
counted_ids = set()

# Vehicle category counts
counts = defaultdict(int)

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=vehicle_classes,
        conf=0.35,
        verbose=False
    )

    result = results[0]

    # Draw counting line
    cv2.line(
        frame,
        (0, line_y),
        (width, line_y),
        (0, 255, 255),
        3
    )

    # Process tracked vehicles
    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.int().cpu().tolist()
        classes = result.boxes.cls.int().cpu().tolist()

        for box, track_id, class_id in zip(boxes, ids, classes):

            x1, y1, x2, y2 = map(int, box)

            # Calculate vehicle centre
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2

            vehicle_name = class_names[class_id]

            # Check whether vehicle crosses counting line
            if track_id in previous_centers:

                previous_y = previous_centers[track_id]

                crossed = (
                    (previous_y < line_y <= center_y) or
                    (previous_y > line_y >= center_y)
                )

                if crossed and track_id not in counted_ids:

                    counts[vehicle_name] += 1
                    counted_ids.add(track_id)

                    print(
                        "Counted:",
                        vehicle_name,
                        "ID:",
                        track_id
                    )

            # Update previous position
            previous_centers[track_id] = center_y

            # Draw bounding box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Display ID
            cv2.putText(
                frame,
                f"{vehicle_name} ID:{track_id}",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

    # Display counts on video
    cv2.putText(
        frame,
        f"Total Count: {sum(counts.values())}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    out.write(frame)

cap.release()
out.release()

print("\nFINAL VEHICLE COUNTS")
print(dict(counts))
print("Total Vehicles:", sum(counts.values()))
print("Saved video:", output_path)


In [ ]:

from IPython.display import Video, display

display(Video(output_path, embed=True))


In [ ]:

import pandas as pd

# Convert vehicle counts into a DataFrame
vehicle_data = pd.DataFrame({
    "Vehicle Type": ["Car", "Motorcycle", "Bus", "Truck"],
    "Count": [
        counts["car"],
        counts["motorcycle"],
        counts["bus"],
        counts["truck"]
    ]
})

# Calculate total
total_vehicles = vehicle_data["Count"].sum()

# Calculate percentage composition
vehicle_data["Percentage (%)"] = (
    vehicle_data["Count"] / total_vehicles * 100
).round(2) if total_vehicles > 0 else 0

print(vehicle_data)
print("\nTotal Vehicles:", total_vehicles)


In [ ]:

# Get video duration in minutes
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

cap.release()

duration_minutes = total_frames / fps / 60

# Calculate observed traffic flow rate
if duration_minutes > 0:
    traffic_volume = total_vehicles / duration_minutes * 60
else:
    traffic_volume = 0

print("Observation Duration:", round(duration_minutes, 2), "minutes")

print(
    "Observed Traffic Flow Rate:",
    round(traffic_volume, 2),
    "vehicles/hour"
)


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.bar(
    vehicle_data["Vehicle Type"],
    vehicle_data["Count"]
)

plt.xlabel("Vehicle Category")
plt.ylabel("Number of Vehicles")
plt.title("AI-Based Vehicle Classification")

plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:

plt.figure(figsize=(8, 8))

plt.pie(
    vehicle_data["Count"],
    labels=vehicle_data["Vehicle Type"],
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Traffic Composition by Vehicle Type")
plt.show()


In [ ]:

# Save CSV
vehicle_data.to_csv(
    "traffic_volume_analysis.csv",
    index=False
)

# Save Excel
with pd.ExcelWriter("traffic_volume_analysis.xlsx") as writer:

    vehicle_data.to_excel(
        writer,
        sheet_name="Vehicle Counts",
        index=False
    )

    summary = pd.DataFrame({
        "Parameter": [
            "Total Vehicles",
            "Observation Duration (minutes)",
            "Hourly Equivalent Flow (vehicles/hour)"
        ],
        "Value": [
            total_vehicles,
            round(duration_minutes, 2),
            round(traffic_volume, 2)
        ]
    })

    summary.to_excel(
        writer,
        sheet_name="Traffic Summary",
        index=False
    )

print("Excel and CSV files created!")


In [ ]:

from google.colab import files

files.download("traffic_volume_analysis.xlsx")
files.download("traffic_volume_analysis.csv")


In [ ]:

import cv2
import pandas as pd
from collections import defaultdict

# Open original video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

line_y = int(height * 0.60)

vehicle_classes = [2, 3, 5, 7]

# Store previous vehicle positions
previous_positions = {}

# Prevent duplicate counting
counted_ids = set()

# Store crossing information
vehicle_events = []

frame_number = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=vehicle_classes,
        conf=0.35,
        verbose=False
    )

    result = results[0]

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.int().cpu().tolist()
        classes = result.boxes.cls.int().cpu().tolist()

        for box, track_id, class_id in zip(boxes, ids, classes):

            x1, y1, x2, y2 = map(int, box)

            center_y = (y1 + y2) // 2

            vehicle_type = model.names[class_id]

            if track_id in previous_positions:

                previous_y = previous_positions[track_id]

                crossed = (
                    previous_y < line_y <= center_y or
                    previous_y > line_y >= center_y
                )

                if crossed and track_id not in counted_ids:

                    direction = (
                        "Downward"
                        if center_y > previous_y
                        else "Upward"
                    )

                    crossing_time = frame_number / fps

                    vehicle_events.append({
                        "Vehicle_ID": track_id,
                        "Vehicle_Type": vehicle_type,
                        "Time_Seconds": crossing_time,
                        "Direction": direction
                    })

                    counted_ids.add(track_id)

            previous_positions[track_id] = center_y

    frame_number += 1

cap.release()

print("Vehicle tracking and event recording completed!")
print("Total crossing events:", len(vehicle_events))


In [ ]:

df = pd.DataFrame(vehicle_events)

if not df.empty:

    df["Time_Minutes"] = df["Time_Seconds"] / 60

    print(df.head(10))

else:
    print("No vehicle crossings detected.")


In [ ]:

if not df.empty:

    df["5_Minute_Interval"] = (
        (df["Time_Seconds"] // 300) * 5
    ).astype(int)

    traffic_5min = df.pivot_table(
        index="5_Minute_Interval",
        columns="Vehicle_Type",
        values="Vehicle_ID",
        aggfunc="count",
        fill_value=0
    )

    traffic_5min["Total_Vehicles"] = (
        traffic_5min.sum(axis=1)
    )

    print(traffic_5min)

else:
    print("No data available.")


In [ ]:

if not df.empty:

    peak_interval = traffic_5min[
        "Total_Vehicles"
    ].idxmax()

    peak_volume = traffic_5min.loc[
        peak_interval, "Total_Vehicles"
    ]

    print("Peak Interval:", peak_interval,
          "to", peak_interval + 5, "minutes")

    print("Maximum Vehicles:", peak_volume)

else:
    print("Insufficient data.")


In [ ]:

if not df.empty:

    plt.figure(figsize=(10, 5))

    plt.plot(
        traffic_5min.index,
        traffic_5min["Total_Vehicles"],
        marker="o"
    )

    plt.xlabel("Time Interval (minutes)")
    plt.ylabel("Number of Vehicles")

    plt.title("5-Minute Traffic Volume Variation")

    plt.grid(True)
    plt.show()


In [ ]:

if not df.empty:

    direction_counts = df.groupby(
        "Direction"
    ).size()

    print(direction_counts)

    direction_counts.plot(
        kind="bar",
        figsize=(7, 5),
        title="Direction-Wise Traffic Volume",
        xlabel="Direction",
        ylabel="Number of Vehicles"
    )

    plt.show()


In [ ]:

if not df.empty:

    df.to_csv(
        "detailed_traffic_data.csv",
        index=False
    )

    with pd.ExcelWriter(
        "detailed_traffic_analysis.xlsx"
    ) as writer:

        df.to_excel(
            writer,
            sheet_name="Vehicle Events",
            index=False
        )

        traffic_5min.to_excel(
            writer,
            sheet_name="5-Minute Volume"
        )

    print("Detailed traffic files created!")

    from google.colab import files

    files.download("detailed_traffic_analysis.xlsx")

else:
    print("No data to export.")


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

# Use actual vehicle crossing records
if df.empty:
    print("No vehicle crossing records available.")
else:

    # Vehicle counts
    vehicle_counts = df["Vehicle_Type"].value_counts()

    # Traffic volume in 5-minute intervals
    df["Interval"] = (
        (df["Time_Seconds"] // 300) * 5
    ).astype(int)

    interval_counts = df.groupby("Interval").size()

    # Direction-wise counts
    direction_counts = df["Direction"].value_counts()

    # Create dashboard
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # Chart 1: Vehicle classification
    vehicle_counts.plot(
        kind="bar",
        ax=axes[0, 0]
    )
    axes[0, 0].set_title("Vehicle Classification")
    axes[0, 0].set_ylabel("Number of Vehicles")

    # Chart 2: Traffic volume
    interval_counts.plot(
        kind="bar",
        ax=axes[0, 1]
    )
    axes[0, 1].set_title("5-Minute Traffic Volume")
    axes[0, 1].set_xlabel("Interval Start (minutes)")

    # Chart 3: Vehicle composition
    vehicle_counts.plot(
        kind="pie",
        autopct="%1.1f%%",
        ax=axes[1, 0]
    )
    axes[1, 0].set_title("Traffic Composition")
    axes[1, 0].set_ylabel("")

    # Chart 4: Direction analysis
    direction_counts.plot(
        kind="bar",
        ax=axes[1, 1]
    )
    axes[1, 1].set_title("Direction-Wise Traffic")
    axes[1, 1].set_ylabel("Number of Vehicles")

    plt.suptitle(
        "AI-Based Traffic Volume Analysis Dashboard",
        fontsize=16
    )

    plt.tight_layout()
    plt.savefig("traffic_dashboard.png", dpi=200)
    plt.show()


In [ ]:

import pandas as pd

# Enter your manually verified counts here
manual_counts = {
    "car": 0,
    "motorcycle": 0,
    "bus": 0,
    "truck": 0
}

# AI counts from our tracking results
ai_counts = df["Vehicle_Type"].value_counts().to_dict()

comparison = pd.DataFrame({
    "Vehicle Type": list(manual_counts.keys()),
    "Manual Count": list(manual_counts.values()),
})

comparison["AI Count"] = comparison["Vehicle Type"].map(
    lambda x: ai_counts.get(x, 0)
)

comparison["Absolute Error"] = abs(
    comparison["AI Count"] - comparison["Manual Count"]
)

comparison["Counting Error (%)"] = comparison.apply(
    lambda row: (
        row["Absolute Error"] / row["Manual Count"] * 100
        if row["Manual Count"] > 0 else None
    ),
    axis=1
)

print(comparison)


In [ ]:

total_manual = comparison["Manual Count"].sum()
total_ai = comparison["AI Count"].sum()

if total_manual > 0:

    overall_error = (
        abs(total_ai - total_manual) / total_manual
    ) * 100

    overall_agreement = 100 - overall_error

    print("Manual Total:", total_manual)
    print("AI Total:", total_ai)
    print("Counting Error:", round(overall_error, 2), "%")
    print("Count Agreement:", round(overall_agreement, 2), "%")

else:
    print("Enter manual counts before calculating.")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(comparison))
bar_width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(
    x - bar_width/2,
    comparison["Manual Count"],
    bar_width,
    label="Manual Count"
)

plt.bar(
    x + bar_width/2,
    comparison["AI Count"],
    bar_width,
    label="AI Count"
)

plt.xticks(x, comparison["Vehicle Type"])
plt.ylabel("Number of Vehicles")
plt.title("Manual vs AI Vehicle Counting")
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.show()


In [ ]:

# STEP 1: Check AI vehicle-counting results

if "df" not in globals() or df.empty:
    print("No vehicle event data found.")
    print("Please rerun the vehicle tracking and line-crossing code from Step 7.")
else:
    print("TOTAL VEHICLES:", len(df))

    print("\nVEHICLE-WISE AI COUNTS:")
    ai_counts = df["Vehicle_Type"].value_counts()
    print(ai_counts)

    print("\nDIRECTION-WISE AI COUNTS:")
    print(df["Direction"].value_counts())

    print("\nFIRST 10 VEHICLE RECORDS:")
    display(df.head(10))


In [ ]:

import pandas as pd

# Provisional manual counts
manual_counts = {
    "car": 23,
    "motorcycle": 5,
    "bus": 0,
    "truck": 0
}

# Get AI counts from our tracking dataset
ai_counts = df["Vehicle_Type"].value_counts().to_dict()

# Create comparison table
comparison = pd.DataFrame({
    "Vehicle Type": list(manual_counts.keys()),
    "Manual Count": list(manual_counts.values())
})

comparison["AI Count"] = comparison["Vehicle Type"].map(
    lambda vehicle: ai_counts.get(vehicle, 0)
)

comparison["Absolute Error"] = (
    comparison["AI Count"] - comparison["Manual Count"]
).abs()

comparison["Counting Error (%)"] = comparison.apply(
    lambda row: (
        row["Absolute Error"] / row["Manual Count"] * 100
        if row["Manual Count"] > 0 else None
    ),
    axis=1
)

display(comparison)


In [ ]:

total_manual = comparison["Manual Count"].sum()
total_ai = comparison["AI Count"].sum()

if total_manual > 0:
    error = abs(total_ai - total_manual) / total_manual * 100
    agreement = 100 - error

    print("Manual Total:", total_manual)
    print("AI Total:", total_ai)
    print("Counting Error:", round(error, 2), "%")
    print("Count Agreement:", round(agreement, 2), "%")


In [ ]:

import matplotlib.pyplot as plt

# Check that the required data exists
if "df" not in globals() or df.empty:
    print("Vehicle data is missing. Run the tracking code first.")

elif "comparison" not in globals():
    print("Run Step 2 first to create the comparison table.")

else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Graph 1: AI vehicle composition
    ai_counts = df["Vehicle_Type"].value_counts()

    axes[0].bar(ai_counts.index, ai_counts.values)
    axes[0].set_title("AI Vehicle Classification")
    axes[0].set_xlabel("Vehicle Type")
    axes[0].set_ylabel("Number of Vehicles")
    axes[0].grid(axis="y", alpha=0.3)

    # Graph 2: Manual vs AI comparison
    x = range(len(comparison))
    width = 0.35

    axes[1].bar(
        [i - width/2 for i in x],
        comparison["Manual Count"],
        width,
        label="Manual"
    )

    axes[1].bar(
        [i + width/2 for i in x],
        comparison["AI Count"],
        width,
        label="AI"
    )

    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(comparison["Vehicle Type"])
    axes[1].set_title("Manual vs AI Vehicle Count")
    axes[1].set_ylabel("Number of Vehicles")
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("vehicle_comparison.png", dpi=300)
    plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Calculate vehicle-wise counts
vehicle_counts = df["Vehicle_Type"].value_counts()

# Calculate total vehicles
total_vehicles = vehicle_counts.sum()

# Calculate percentage composition
composition = pd.DataFrame({
    "Vehicle Type": vehicle_counts.index,
    "Vehicle Count": vehicle_counts.values
})

composition["Percentage (%)"] = (
    composition["Vehicle Count"] / total_vehicles
) * 100

display(composition.round(2))


In [ ]:

plt.figure(figsize=(7, 7))

plt.pie(
    vehicle_counts.values,
    labels=vehicle_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Vehicle Composition in Observed Traffic")
plt.savefig("vehicle_composition.png", dpi=300)
plt.show()


In [ ]:

# STEP 5: Traffic Flow Rate

# Observation duration from your video
duration_seconds = 16.1

# Number of vehicles counted by AI
total_vehicles = len(df)

# Calculate flow rate
flow_rate = (total_vehicles / duration_seconds) * 3600

print("Observation Duration:", duration_seconds, "seconds")
print("Total AI Vehicles:", total_vehicles)
print("Traffic Flow Rate:", round(flow_rate, 2), "vehicles/hour")

# Calculate flow rate for each vehicle category
print("\nVehicle-wise Flow Rate:")

for vehicle_type, count in df["Vehicle_Type"].value_counts().items():
    rate = (count / duration_seconds) * 3600
    print(vehicle_type, ":", round(rate, 2), "vehicles/hour")


In [ ]:

import matplotlib.pyplot as plt

vehicle_counts = df["Vehicle_Type"].value_counts()

flow_by_type = vehicle_counts / duration_seconds * 3600

plt.figure(figsize=(9, 5))
plt.bar(flow_by_type.index, flow_by_type.values)

plt.title("Estimated Hourly-Equivalent Traffic Flow by Vehicle Type")
plt.xlabel("Vehicle Type")
plt.ylabel("Vehicles per Hour")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig("traffic_flow_rate.png", dpi=300)
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. PREPARE PROJECT DATA
# ==========================================

if "df" not in globals() or df.empty:
    raise ValueError(
        "Vehicle dataset is empty. Run the tracking code first."
    )

duration_seconds = 16.1

vehicle_counts = df["Vehicle_Type"].value_counts()
total_vehicles = len(df)

flow_rate = (
    total_vehicles / duration_seconds
) * 3600

# Direction data, if available
if "Direction" in df.columns:
    direction_counts = df["Direction"].value_counts()
else:
    direction_counts = pd.Series(dtype=int)

# Manual comparison, if available
has_comparison = (
    "comparison" in globals()
    and not comparison.empty
)

if has_comparison:
    total_manual = comparison["Manual Count"].sum()
    total_ai = comparison["AI Count"].sum()

    if total_manual > 0:
        count_agreement = (
            100 - abs(total_ai - total_manual)
            / total_manual * 100
        )
    else:
        count_agreement = None
else:
    count_agreement = None


# ==========================================
# 2. CREATE DASHBOARD LAYOUT
# ==========================================

fig = plt.figure(figsize=(16, 11))

fig.suptitle(
    "AI-BASED TRAFFIC ANALYTICS DASHBOARD",
    fontsize=21,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.5, 0.945,
    "Vehicle Detection | Tracking | Traffic Volume Analysis",
    ha="center",
    fontsize=11
)

gs = fig.add_gridspec(
    3, 2,
    top=0.89,
    bottom=0.08,
    hspace=0.42,
    wspace=0.28
)


# ==========================================
# 3. KPI SUMMARY
# ==========================================

kpi_ax = fig.add_subplot(gs[0, :])
kpi_ax.axis("off")

agreement_text = (
    f"{count_agreement:.1f}%"
    if count_agreement is not None
    else "Not validated"
)

kpis = [
    ("TOTAL VEHICLES", str(total_vehicles)),
    ("CATEGORIES", str(len(vehicle_counts))),
    ("OBSERVATION", f"{duration_seconds} sec"),
    ("FLOW RATE", f"{flow_rate:.0f} veh/hr"),
    ("COUNT AGREEMENT", agreement_text)
]

positions = np.linspace(0.1, 0.9, len(kpis))

for x, (label, value) in zip(positions, kpis):
    kpi_ax.text(
        x, 0.65, value,
        ha="center",
        fontsize=19,
        fontweight="bold",
        transform=kpi_ax.transAxes
    )

    kpi_ax.text(
        x, 0.25, label,
        ha="center",
        fontsize=9,
        transform=kpi_ax.transAxes
    )


# ==========================================
# 4. VEHICLE COMPOSITION
# ==========================================

ax1 = fig.add_subplot(gs[1, 0])

ax1.pie(
    vehicle_counts.values,
    labels=vehicle_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

ax1.set_title(
    "Vehicle Composition",
    fontsize=14,
    fontweight="bold"
)


# ==========================================
# 5. VEHICLE CATEGORY COUNTS
# ==========================================

ax2 = fig.add_subplot(gs[1, 1])

bars = ax2.bar(
    vehicle_counts.index,
    vehicle_counts.values
)

ax2.bar_label(bars)

ax2.set_title(
    "Vehicle Classification",
    fontsize=14,
    fontweight="bold"
)

ax2.set_xlabel("Vehicle Type")
ax2.set_ylabel("Number of Vehicles")
ax2.grid(axis="y", alpha=0.3)


# ==========================================
# 6. MANUAL VS AI COMPARISON
# ==========================================

ax3 = fig.add_subplot(gs[2, 0])

if has_comparison:
    x = np.arange(len(comparison))
    width = 0.35

    ax3.bar(
        x - width/2,
        comparison["Manual Count"],
        width,
        label="Manual"
    )

    ax3.bar(
        x + width/2,
        comparison["AI Count"],
        width,
        label="AI"
    )

    ax3.set_xticks(x)
    ax3.set_xticklabels(comparison["Vehicle Type"])
    ax3.legend()

    ax3.set_title(
        "Manual vs AI Count (Provisional)",
        fontsize=14,
        fontweight="bold"
    )

    ax3.set_ylabel("Vehicle Count")
    ax3.grid(axis="y", alpha=0.3)

else:
    ax3.text(
        0.5, 0.5,
        "Manual validation data unavailable",
        ha="center",
        va="center",
        transform=ax3.transAxes
    )

    ax3.set_title("Manual vs AI Comparison")


# ==========================================
# 7. DIRECTION-WISE TRAFFIC
# ==========================================

ax4 = fig.add_subplot(gs[2, 1])

if not direction_counts.empty:
    bars = ax4.bar(
        direction_counts.index.astype(str),
        direction_counts.values
    )

    ax4.bar_label(bars)

    ax4.set_title(
        "Direction-Wise Traffic",
        fontsize=14,
        fontweight="bold"
    )

    ax4.set_ylabel("Number of Vehicles")
    ax4.grid(axis="y", alpha=0.3)

else:
    ax4.text(
        0.5, 0.5,
        "Direction data unavailable",
        ha="center",
        va="center",
        transform=ax4.transAxes
    )

    ax4.set_title("Direction-Wise Traffic")


# ==========================================
# 8. FOOTER AND EXPORT
# ==========================================

fig.text(
    0.5, 0.025,
    "Technology: YOLO + ByteTrack + OpenCV + Python",
    ha="center",
    fontsize=10
)

plt.savefig(
    "Final_Traffic_Analytics_Dashboard.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("FINAL DASHBOARD GENERATED SUCCESSFULLY!")


In [ ]:

from google.colab import files

files.download(
    "Final_Traffic_Analytics_Dashboard.png"
)


In [ ]:

# Save detailed vehicle records
df.to_csv(
    "Final_Vehicle_Dataset.csv",
    index=False
)

# Save Excel workbook
with pd.ExcelWriter(
    "Final_Traffic_Analysis.xlsx"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Vehicle Records",
        index=False
    )

    vehicle_counts.rename(
        "Vehicle Count"
    ).to_excel(
        writer,
        sheet_name="Vehicle Composition"
    )

    if has_comparison:
        comparison.to_excel(
            writer,
            sheet_name="Manual vs AI",
            index=False
        )

print("Project datasets exported!")

files.download("Final_Vehicle_Dataset.csv")
files.download("Final_Traffic_Analysis.xlsx")


In [ ]:

from ultralytics import YOLO
import cv2
from collections import defaultdict
from google.colab.patches import cv2_imshow

# Load YOLO model
model = YOLO("yolo11n.pt")

# Vehicle classes: car, motorcycle, bus, truck
vehicle_classes = [2, 3, 5, 7]

# Open original video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

line_y = int(height * 0.60)

previous_positions = {}
counted_ids = set()
vehicle_events = []

counts = defaultdict(int)
frame_number = 0

class_names = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=vehicle_classes,
        conf=0.35,
        verbose=False
    )

    annotated = frame.copy()

    cv2.line(
        annotated,
        (0, line_y),
        (width, line_y),
        (0, 255, 0),
        3
    )

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls in zip(boxes, ids, classes):

            x1, y1, x2, y2 = map(int, box)

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            if track_id in previous_positions:

                previous_y = previous_positions[track_id]

                crossed = (
                    (previous_y < line_y <= center_y)
                    or
                    (previous_y > line_y >= center_y)
                )

                if crossed and track_id not in counted_ids:

                    vehicle_type = class_names[cls]

                    counted_ids.add(track_id)
                    counts[vehicle_type] += 1

                    vehicle_events.append({
                        "Vehicle_ID": int(track_id),
                        "Vehicle_Type": vehicle_type,
                        "Time_Seconds": frame_number / fps,
                        "Direction": (
                            "Downward"
                            if center_y > previous_y
                            else "Upward"
                        )
                    })

            previous_positions[track_id] = center_y

            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2
            )

            cv2.putText(
                annotated,
                f"ID: {track_id}",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                2
            )

    cv2.putText(
        annotated,
        f"Total Count: {len(counted_ids)}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    frame_number += 1

cap.release()

df = pd.DataFrame(vehicle_events)

print("FINAL AI COUNTS:", dict(counts))
print("TOTAL VEHICLES:", len(counted_ids))
print("VEHICLE EVENTS:", len(vehicle_events))


In [ ]:

from ultralytics import YOLO
import cv2
from IPython.display import Video, display

# Load model and original video
model = YOLO("yolo11n.pt")
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if not fps or fps <= 0:
    fps = 30

line_y = int(height * 0.60)
output_path = "/content/annotated_traffic_video.mp4"

# Create output video
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

vehicle_classes = [2, 3, 5, 7]
frame_count = 0

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=vehicle_classes,
        conf=0.35,
        verbose=False
    )

    annotated = results[0].plot()

    # Draw counting line
    cv2.line(
        annotated,
        (0, line_y),
        (width, line_y),
        (0, 255, 0),
        3
    )

    out.write(annotated)
    frame_count += 1

cap.release()
out.release()

print("Annotated video created!")
print("Frames processed:", frame_count)
print("Output:", output_path)

display(Video(output_path, embed=True))


In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/annotated_traffic_video.mp4",
    embed=True,
    width=800
))

In [ ]:

from collections import Counter
import pandas as pd

# Count vehicle classifications from the latest AI run
class_counts = Counter()

cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.predict(
        frame,
        classes=[2, 3, 5, 7],
        conf=0.35,
        verbose=False
    )

    for cls in results[0].boxes.cls.cpu().numpy().astype(int):
        class_counts[cls] += 1

cap.release()

class_names = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck"
}

classification_df = pd.DataFrame([
    {"Vehicle Type": class_names[k], "Frame Detections": v}
    for k, v in class_counts.items()
])

display(classification_df)


In [ ]:

import cv2
from google.colab.patches import cv2_imshow

cap = cv2.VideoCapture(video_path)

shown = 0
frame_number = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success or shown >= 6:
        break

    results = model.predict(
        frame,
        classes=[3],
        conf=0.15,
        verbose=False
    )

    if len(results[0].boxes) > 0:
        annotated = results[0].plot()

        print("Motorcycle detection in frame:", frame_number)
        cv2_imshow(annotated)

        shown += 1

    frame_number += 1

cap.release()

print("Motorcycle detection frames displayed:", shown)


In [ ]:

from collections import defaultdict, Counter
from ultralytics import YOLO
import cv2
import pandas as pd

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture(video_path)

vehicle_classes = [2, 3, 5, 7]

class_names = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck"
}

track_classes = defaultdict(Counter)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=vehicle_classes,
        conf=0.25,
        verbose=False
    )

    boxes = results[0].boxes

    if boxes.id is not None:
        ids = boxes.id.cpu().numpy().astype(int)
        classes = boxes.cls.cpu().numpy().astype(int)

        for track_id, cls in zip(ids, classes):
            track_classes[track_id][class_names[cls]] += 1

cap.release()

summary = []

for track_id, class_counter in track_classes.items():
    most_common_class = class_counter.most_common(1)[0][0]

    summary.append({
        "Tracking_ID": track_id,
        "Vehicle_Type": most_common_class,
        "Frames_Tracked": sum(class_counter.values())
    })

tracking_df = pd.DataFrame(summary)

print("Unique tracking IDs:", len(tracking_df))
display(tracking_df.sort_values("Tracking_ID"))


In [ ]:

# Analyse how long each tracking ID survived

print("Total tracking IDs:", len(tracking_df))

print("\nTracking duration categories:")

duration_summary = pd.Series({
    "1–5 frames": ((tracking_df["Frames_Tracked"] >= 1) &
                   (tracking_df["Frames_Tracked"] <= 5)).sum(),

    "6–15 frames": ((tracking_df["Frames_Tracked"] >= 6) &
                    (tracking_df["Frames_Tracked"] <= 15)).sum(),

    "16–30 frames": ((tracking_df["Frames_Tracked"] >= 16) &
                     (tracking_df["Frames_Tracked"] <= 30)).sum(),

    "More than 30 frames": (tracking_df["Frames_Tracked"] > 30).sum()
})

display(duration_summary.to_frame("Number of Tracks"))

print("\nAverage tracking duration:",
      round(tracking_df["Frames_Tracked"].mean(), 2),
      "frames")


In [ ]:

from collections import defaultdict, Counter
from ultralytics import YOLO
import cv2
import pandas as pd

model = YOLO("yolo11n.pt")
cap = cv2.VideoCapture(video_path)

track_classes_v2 = defaultdict(Counter)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[2, 3, 5, 7],
        conf=0.20,
        verbose=False
    )

    boxes = results[0].boxes

    if boxes.id is not None:
        ids = boxes.id.cpu().numpy().astype(int)
        classes = boxes.cls.cpu().numpy().astype(int)

        for track_id, cls in zip(ids, classes):
            track_classes_v2[track_id][int(cls)] += 1

cap.release()

class_names = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck"
}

summary_v2 = []

for track_id, class_counter in track_classes_v2.items():
    cls = class_counter.most_common(1)[0][0]

    summary_v2.append({
        "Tracking_ID": track_id,
        "Vehicle_Type": class_names[cls],
        "Frames_Tracked": sum(class_counter.values())
    })

tracking_df_v2 = pd.DataFrame(summary_v2)

print("Original tracking IDs:", len(tracking_df))
print("New tracking IDs:", len(tracking_df_v2))

display(
    tracking_df_v2["Vehicle_Type"]
    .value_counts()
    .to_frame("Number of Tracks")
)


In [ ]:

duration_summary_v2 = pd.Series({
    "1–5 frames": (
        (tracking_df_v2["Frames_Tracked"] >= 1) &
        (tracking_df_v2["Frames_Tracked"] <= 5)
    ).sum(),

    "6–15 frames": (
        (tracking_df_v2["Frames_Tracked"] >= 6) &
        (tracking_df_v2["Frames_Tracked"] <= 15)
    ).sum(),

    "16–30 frames": (
        (tracking_df_v2["Frames_Tracked"] >= 16) &
        (tracking_df_v2["Frames_Tracked"] <= 30)
    ).sum(),

    "More than 30 frames": (
        tracking_df_v2["Frames_Tracked"] > 30
    ).sum()
})

display(duration_summary_v2.to_frame("Number of Tracks"))


In [ ]:

print("Original crossing events:", len(df))

if not df.empty:
    print("\nVehicle counts:")
    display(df["Vehicle_Type"].value_counts().to_frame("Count"))

    print("\nDirection counts:")
    display(df["Direction"].value_counts().to_frame("Count"))

    print("\nVehicle event records:")
    display(df.sort_values("Time_Seconds"))


In [ ]:

# Traffic flow analysis

video_duration = 482 / 30  # seconds
total_vehicles = len(df)

flow_rate_per_second = total_vehicles / video_duration
hourly_equivalent = flow_rate_per_second * 3600

vehicle_counts = df["Vehicle_Type"].value_counts()

print("VIDEO DURATION:", round(video_duration, 2), "seconds")
print("OBSERVED CROSSINGS:", total_vehicles)
print("OBSERVED FLOW:", round(flow_rate_per_second, 3), "vehicles/second")
print("HOURLY-EQUIVALENT FLOW:", round(hourly_equivalent, 2), "vehicles/hour")

print("\nVEHICLE COMPOSITION:")
for vehicle, count in vehicle_counts.items():
    print(vehicle, ":", count,
          "(", round(count / total_vehicles * 100, 2), "%)")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Prepare dashboard data
video_duration = 16.07
total = len(df)

counts = df["Vehicle_Type"].value_counts()
vehicle_types = ["car", "motorcycle", "bus", "truck"]
counts = counts.reindex(vehicle_types, fill_value=0)

flow = total / video_duration
hourly_equivalent = flow * 3600

# Create dashboard
fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    "AI-BASED TRAFFIC ANALYTICS DASHBOARD",
    fontsize=20, fontweight="bold"
)

gs = fig.add_gridspec(3, 2, height_ratios=[0.8, 2, 1.5])

# KPI cards
kpis = [
    ("TOTAL CROSSINGS", str(total)),
    ("VIDEO DURATION", f"{video_duration:.2f} sec"),
    ("OBSERVED FLOW", f"{flow:.3f} veh/sec"),
    ("HOURLY EQUIVALENT", f"{hourly_equivalent:.0f} veh/hr")
]

for i, (label, value) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i % 2])
    if i >= 2:
        ax = fig.add_subplot(gs[1, i - 2])
    ax.axis("off")
    ax.text(0.5, 0.68, value, ha="center",
            fontsize=23, fontweight="bold")
    ax.text(0.5, 0.25, label, ha="center", fontsize=10)

# Vehicle composition
ax1 = fig.add_subplot(gs[1, 0])
nonzero = counts[counts > 0]
ax1.pie(
    nonzero.values,
    labels=nonzero.index,
    autopct="%1.1f%%",
    startangle=90
)
ax1.set_title("Vehicle Composition")

# Vehicle category bar chart
ax2 = fig.add_subplot(gs[1, 1])
ax2.bar(counts.index, counts.values)
ax2.set_title("Crossings by Vehicle Type")
ax2.set_ylabel("Number of crossings")

# Crossing timeline
ax3 = fig.add_subplot(gs[2, :])
if not df.empty:
    ax3.scatter(
        df["Time_Seconds"],
        np.ones(len(df)),
        s=65
    )
    ax3.set_yticks([])
    ax3.set_xlabel("Time in video (seconds)")
    ax3.set_title("Vehicle Crossing Timeline")
    ax3.set_xlim(0, video_duration)

fig.text(
    0.5, 0.015,
    "Prototype AI results | 16.07-second sample | "
    "Hourly equivalent is extrapolated, not measured traffic volume. "
    "Vehicle classifications and counts require manual validation.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.05, 1, 0.94])

plt.savefig(
    "Portfolio_Traffic_Dashboard.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Dashboard saved successfully!")


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

plt.close("all")

# 1. Prepare data
video_duration = 16.07
total = len(df)

vehicle_types = ["car", "motorcycle", "bus", "truck"]
counts = df["Vehicle_Type"].value_counts().reindex(
    vehicle_types, fill_value=0
)

flow = total / video_duration
hourly_equivalent = flow * 3600

# 2. Create clean layout
fig = plt.figure(figsize=(16, 10))

gs = fig.add_gridspec(
    3, 4,
    height_ratios=[1, 3, 2],
    hspace=0.38,
    wspace=0.35
)

fig.suptitle(
    "AI-BASED VEHICLE DETECTION & TRAFFIC ANALYTICS",
    fontsize=19,
    fontweight="bold",
    y=0.98
)

# 3. Four KPI cards
kpis = [
    ("TOTAL CROSSINGS", f"{total}"),
    ("VIDEO DURATION", f"{video_duration:.2f} sec"),
    ("OBSERVED FLOW", f"{flow:.3f} veh/sec"),
    ("HOURLY EQUIVALENT*", f"{hourly_equivalent:.0f} veh/hr")
]

for i, (label, value) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.axis("off")

    ax.text(
        0.5, 0.62, value,
        ha="center", va="center",
        fontsize=21, fontweight="bold",
        transform=ax.transAxes
    )

    ax.text(
        0.5, 0.22, label,
        ha="center", va="center",
        fontsize=10,
        transform=ax.transAxes
    )

    for spine in ax.spines.values():
        spine.set_visible(True)

# 4. Vehicle composition
ax1 = fig.add_subplot(gs[1, 0:2])

nonzero = counts[counts > 0]

ax1.pie(
    nonzero.values,
    labels=nonzero.index,
    autopct="%1.1f%%",
    startangle=90
)

ax1.set_title("Vehicle Composition", fontweight="bold")

# 5. Vehicle category chart
ax2 = fig.add_subplot(gs[1, 2:4])

bars = ax2.bar(counts.index, counts.values)
ax2.bar_label(bars, padding=3)

ax2.set_title("Vehicle Crossings by Category", fontweight="bold")
ax2.set_ylabel("Number of crossings")
ax2.set_ylim(0, max(counts.max() + 5, 5))

# 6. Crossing timeline
ax3 = fig.add_subplot(gs[2, :])

ax3.scatter(
    df["Time_Seconds"],
    np.ones(total),
    s=55
)

ax3.set_xlim(0, video_duration)
ax3.set_ylim(0.8, 1.2)
ax3.set_yticks([])
ax3.set_xlabel("Time (seconds)")
ax3.set_title("Vehicle Crossing Timeline", fontweight="bold")
ax3.grid(axis="x", alpha=0.3)

# 7. Footer
fig.text(
    0.5, 0.015,
    "*Hourly equivalent is extrapolated from a 16.07-second "
    "sample, not measured hourly traffic. AI classifications "
    "and counts require manual validation.",
    ha="center",
    fontsize=9
)

plt.savefig(
    "FINAL_Portfolio_Traffic_Dashboard.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Professional dashboard saved successfully!")
